# ML-LPDS initialisation: `level` vs `cascade` normalisation

`spectral_normalize` sets `||A_l||_2 = 1` **per level**, measured on *generic*
level-(l-1) input. The operator the algorithm actually applies is the cascade
`A_(1,l) = A_l ... A_1`, and for independently normalised factors that product
falls well below 1 -- their top singular directions do not align, and `A_l` is
normalised against inputs it never receives (in the net it only ever sees
`A_(1,l-1) x`, which lives in the range of the previous cascade).

Measured at init on the `mllpdsw2` shape, with every `||A_l|| ~ 0.99`:

    ||A_(1,1)|| = 0.99    ||A_(1,2)|| = 0.48    ...

So level 2's dual receives half the gain level 1's does while `lam0` clips both
against the same threshold, and its push back into the primal carries
`||A_(1,l)||` a second time. `init_norm='cascade'` rescales each level so every
`||A_(1,l)|| = 1`.

**The cost is the depth tax.** `||A||^2` rises from ~1.1 toward `L`, so the
admissible Condat-Vu step `1/(1/2 + ||A||^2)` falls. `TAU0_SCALE` below halves
`tau0` for the cascade arm; set it to 1.0 to see what happens if you don't.

This notebook does three things: reads the init diagnostics (no training),
checks `project()` does not undo the rescale, then trains every arm briefly on
fastMRI brain and compares. **A few hundred steps is not a verdict** -- it says
which arm starts better and whether any of them is broken, nothing more.

In [ ]:
import json, os, pathlib, sys, copy, time
import matplotlib.pyplot as plt
import torch

ROOT = pathlib.Path.cwd()
if not (ROOT / "physics").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from datasets.registry import build_loader
from models import build_model
from operators import Mask, FFT2D, Sense
from physics.mask import get_mask_cached as get_mask
from training.common import prepare_measurement, embed_for_net, apply_loss_mask
from training.losses import LOSS_REGISTRY
from training.metrics import compute_metrics

# ============================ KNOBS ============================
CONFIG      = "config/brain/mg/mllpds_R16.json"   # shape, data and MRI settings
ARMS        = [("level", "slice"),                # (init_norm, proj_mode)
               ("cascade", "slice"),
               ("level", "atom"),
               ("cascade", "atom")]
TAU0_SCALE  = {"level": 1.0, "cascade": 0.5}      # cascade raises ||A||^2 -> halve the step
STEPS       = 300                                 # gradient steps per arm
VAL_EVERY   = 25
VAL_BATCHES = 4
SEED        = 0
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
# ===============================================================

cfg = json.load(open(CONFIG))
mri, tr = cfg["mri"], cfg["training"]
print(f"{CONFIG}\n  model  {cfg['model']['params']}")
print(f"  mri    R={mri['R']} acs={mri['acs_lines']} {mri['kspace_type']}")
print(f"  noise  {tr['noise_std']} ({tr['noise_dist']})   loss {tr['loss_type']}")
print(f"  device {DEVICE}")

## 1) Init diagnostics -- no training

`cascade_norms()` is `[||A_(1,1)||, ..., ||A_(1,L)||]`, exact at init because
`B_l = A_l^H` there. Under `level` these should decay with depth; under
`cascade` they should all be ~1.

`||A||^2` is the stacked operator norm that enters the Condat-Vu bound, and
`step_bound = 1/(1/2 + ||A||^2)` is the largest admissible `tau`. Compare it
against the `tau0` each arm will actually train with.

In [ ]:
def make_net(init_norm, proj_mode, device=DEVICE):
    c = copy.deepcopy(cfg)
    p = c["model"]["params"]
    p["init_norm"], p["proj_mode"] = init_norm, proj_mode
    p["tau0"] = p.get("tau0", 0.1) * TAU0_SCALE[init_norm]
    torch.manual_seed(SEED)                     # same filter draw for every arm
    return build_model(c).to(device), p["tau0"]

def atom_norms(net, k=-1):
    """max ||a_m|| per level -- what the two projections actually bound."""
    return [float(l.analysis.weight.abs().pow(2).sum(dim=(1, 2, 3)).sqrt().max())
            for l in net.layer(k).levels]

rows = {}
for init_norm, proj_mode in ARMS:
    net, tau0 = make_net(init_norm, proj_mode)
    cn = net.cascade_norms(size=64, num_iter=100)
    a2 = net.layer().op_norm2(size=64, num_iter=100)
    rows[(init_norm, proj_mode)] = dict(
        cascade=cn, opnorm2=a2, bound=1.0 / (0.5 + a2), tau0=tau0,
        atoms=atom_norms(net))
    del net

print("%-9s %-6s %-26s %8s %9s %7s  %s"
      % ("init_norm", "proj", "||A_(1,l)||", "||A||^2", "bound", "tau0", "max||a_m||"))
for (i, p), r in rows.items():
    print("%-9s %-6s %-26s %8.3f %9.3f %7.3f  %s"
          % (i, p, " ".join("%.3f" % v for v in r["cascade"]),
             r["opnorm2"], r["bound"], r["tau0"],
             " ".join("%.2f" % v for v in r["atoms"])))
print("\\ntau0 over bound (>1 means outside Condat-Vu at init):")
for (i, p), r in rows.items():
    print("  %-9s %-6s %.2f" % (i, p, r["tau0"] / r["bound"]))

## 2) Does `project()` undo the rescale?

`normalize_cascade` scales level *l* up by `1/||A_(1,l)||`, which also scales its
filter norms up. If that pushes slices past the unit ball, the first
`project()` claws it back and the option is inert. This is the check --
`cascade_norms` should barely move.

In [ ]:
for init_norm, proj_mode in ARMS:
    net, _ = make_net(init_norm, proj_mode)
    before = net.cascade_norms(size=64, num_iter=100)
    net.project()
    after = net.cascade_norms(size=64, num_iter=100)
    drop = max(abs(b - a) for b, a in zip(before, after))
    print("%-9s %-6s  before %s  ->  after %s   max drop %.4f"
          % (init_norm, proj_mode,
             " ".join("%.3f" % v for v in before),
             " ".join("%.3f" % v for v in after), drop))
    del net

## 3) Brief training on fastMRI brain

Same pipeline as `training/recon.py` -- `get_mask` -> `prepare_measurement` ->
`E = Mask @ FFT2D @ Sense` -> `embed_for_net`, the config's own loss and MRI
settings. Every arm sees the **same** mask and noise realisation at each step,
so the curves differ because the model differs.

Validation uses a fixed batch and a fixed generator seed, for the same reason.

In [ ]:
def loaders():
    d = dict(cfg["data"]["train"]); d["batch_size"] = 1
    v = dict(cfg["data"]["val"]);   v["batch_size"] = 1
    return (build_loader(d, shuffle=True, drop_last=True),
            build_loader(v, shuffle=False, drop_last=False))

train_loader, val_loader = loaders()
val_batches = [b for _, b in zip(range(VAL_BATCHES), val_loader)]
loss_fn = LOSS_REGISTRY[tr["loss_type"]]
USE_MASK = tr.get("use_organ_mask", False)

def measurement(batch, device, gen=None):
    # The recon loader yields FIVE items -- `pad_hw` is the embedded size it
    # chose from its own `pad_multiple`, and `embed_for_net` needs exactly that
    # one (re-deriving it from the config would silently score a different
    # operator than training validated on).
    kspace, smaps, image, organ_mask, pad_hw = batch
    kspace, smaps, image, organ_mask = [t.to(device) for t in
                                        (kspace, smaps, image, organ_mask)]
    mask = get_mask(image, R=mri["R"], acs_lines=mri["acs_lines"],
                    mode=mri["mask_dist"], offset=mri.get("mask_offset", 0))
    y, sigma_n, extra = prepare_measurement(
        image=image, kspace=kspace, mask=mask, smaps=smaps,
        kspace_type=mri["kspace_type"], noise_std=tr["noise_std"],
        noise_dist=tr["noise_dist"], whiten_kspace=mri.get("whiten_kspace", False),
        generator=gen)
    E = Mask(mask) @ FFT2D() @ Sense(extra["smaps"])
    return y, sigma_n, E, image, organ_mask, pad_hw

@torch.no_grad()
def validate(net):
    net.eval()
    tot, psnr, n = 0.0, 0.0, 0
    for b in val_batches:
        g = torch.Generator(device=DEVICE); g.manual_seed(tr.get("val_seed", 1234))
        y, s, E, image, om, pad_hw = measurement(b, DEVICE, gen=g)
        Ee, T = embed_for_net(net, E, image, pad_hw)
        rec = T.forward(net(y, E=Ee, sigma=s)[0])
        il, rl = apply_loss_mask(image, rec, om, USE_MASK)
        tot += float(loss_fn(il, rl, s))
        m = compute_metrics(image, rec, mask=om if USE_MASK else None)
        psnr += float(m["psnr"]); n += 1
    net.train()
    return tot / n, psnr / n

def run(init_norm, proj_mode):
    net, tau0 = make_net(init_norm, proj_mode)
    opt = torch.optim.Adam(net.parameters(), **cfg["optimizer"]["params"])
    hist, it, t0 = [], iter(train_loader), time.time()
    for step in range(STEPS + 1):
        if step % VAL_EVERY == 0:
            hist.append((step,) + validate(net))
            print("    step %4d  val_loss %.5f  psnr %.2f" % hist[-1], flush=True)
        try:
            batch = next(it)
        except StopIteration:
            it = iter(train_loader); batch = next(it)
        torch.manual_seed(SEED * 100000 + step)     # same draw across arms
        y, s, E, image, om, pad_hw = measurement(batch, DEVICE)
        Ee, T = embed_for_net(net, E, image, pad_hw)
        opt.zero_grad(set_to_none=True)
        rec = T.forward(net(y, E=Ee, sigma=s)[0])
        il, rl = apply_loss_mask(image, rec, om, USE_MASK)
        loss = loss_fn(il, rl, s)
        if not torch.isfinite(loss):
            print("    step %4d  NON-FINITE loss -- stopping this arm" % step)
            break
        loss.backward()
        if tr.get("clip_grad"):
            torch.nn.utils.clip_grad_norm_(net.parameters(), tr["clip_grad"])
        opt.step()
        if hasattr(net, "project"):
            net.project()
    return hist, net, time.time() - t0

results = {}
for init_norm, proj_mode in ARMS:
    print("== init_norm=%s  proj_mode=%s ==" % (init_norm, proj_mode), flush=True)
    hist, net, secs = run(init_norm, proj_mode)
    results[(init_norm, proj_mode)] = dict(
        hist=hist, secs=secs,
        cascade_end=net.cascade_norms(size=64, num_iter=100),
        opnorm2_end=net.layer().op_norm2(size=64, num_iter=100),
        tau_end=float(net.layer().tau.weight.abs().max()))
    print("   %.0fs" % secs, flush=True)
    del net
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 4) Results

The left panel is the number that matters. The right one says whether the
cascade gains held up under training -- if `level` arms end with the deep
levels still collapsed, that is the mechanism doing its work.

`tau_end` against `step_bound` says whether the learned step walked outside
Condat-Vu, which is the separate question from earlier.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for (i, p), r in results.items():
    st = [h[0] for h in r["hist"]]
    ax[0].plot(st, [h[1] for h in r["hist"]], marker="o", ms=3, label=f"{i}/{p}")
    ax[1].plot(st, [h[2] for h in r["hist"]], marker="o", ms=3, label=f"{i}/{p}")
ax[0].set_xlabel("step"); ax[0].set_ylabel("val loss"); ax[0].set_title("validation loss")
ax[1].set_xlabel("step"); ax[1].set_ylabel("PSNR (dB)"); ax[1].set_title("validation PSNR")
for a in ax: a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("%-9s %-6s %10s %8s  %-24s %9s %8s"
      % ("init_norm", "proj", "val_loss", "psnr", "||A_(1,l)|| after", "||A||^2", "tau_end"))
for (i, p), r in results.items():
    s, l, ps = r["hist"][-1]
    print("%-9s %-6s %10.5f %8.2f  %-24s %9.3f %8.3f"
          % (i, p, l, ps, " ".join("%.3f" % v for v in r["cascade_end"]),
             r["opnorm2_end"], r["tau_end"]))
print("\\ntau_end vs bound 1/(1/2+||A||^2):")
for (i, p), r in results.items():
    b = 1.0 / (0.5 + r["opnorm2_end"])
    print("  %-9s %-6s tau=%.3f bound=%.3f ratio=%.2f" % (i, p, r["tau_end"], b, r["tau_end"] / b))

### Reading this

* **`cascade` should start ahead** if the collapse is real -- the deep levels are
  carrying signal from step 0 instead of having to be grown.
* **If `cascade` is worse**, the most likely cause is the step size: `||A||^2`
  went up and `TAU0_SCALE` did not compensate enough. Try `0.3` before
  concluding the normalisation is wrong.
* **`atom` should only matter at levels >= 2** -- it is provably a no-op at
  `in_channels = 1`, so any level-1 difference is noise.
* A few hundred steps ranks *starting behaviour*. It cannot tell you which arm
  wins at 6000 epochs, and an arm that starts behind may still end ahead.